# NB11B — Domain-aware historical classifier + group-level resolution

Bản sửa này giải quyết hai lỗi phương pháp của NB11 historical cũ:

1. **Không đánh giá trên easy-source mixture.** Confirmed duplicates chỉ là training anchors; validation/test chỉ gồm `hard_negative_review`, và mọi E3 group của validation/test đều bị loại khỏi train.
2. **Không gọi active-learning query là test.** Các `round_*.xlsx` hiện tại là những mẫu model chủ động chọn vì khó; sau khi human-label, chúng được đưa trở lại TRAIN với weight cao.

Notebook cũng tách hai queue:

- `uncertainty query`: dùng để cải thiện classifier;
- `resolution query`: dùng để hoàn tất audit nhanh, chọn candidate có P(DUP) cao nhất và tối đa một pair cho mỗi E3 group chưa được xác nhận duplicate.

`SAME_PRODUCT_DIFFERENT_IMAGE` được giữ lại trong audit nhưng map thành binary `DUPLICATE=1` theo policy hiện tại của project.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive, auth
    drive.mount('/content/drive', force_remount=False)
    auth.authenticate_user()
except ImportError:
    pass

REPO_URL='https://github.com/ThinhTran2208/opisoverated.git'
BRANCH='fix/evaluation3-nb11-domain-aware'
REPO_ROOT=Path('/content/opisoverated-e3-nb11b')
def run_git(*args,cwd=None):
    return subprocess.run(['git','-c','http.version=HTTP/1.1',*args],cwd=cwd,check=True,text=True)
if not (REPO_ROOT/'.git').is_dir():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    run_git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_ROOT))
else:
    run_git('fetch','origin',BRANCH,cwd=REPO_ROOT)
    run_git('switch',BRANCH,cwd=REPO_ROOT)
    run_git('pull','--ff-only','origin',BRANCH,cwd=REPO_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO_ROOT/'requirements-evaluation.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','gspread>=6,<7'],check=True)
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))

from src.evaluation.evaluation3_historical_classifier import (
    CONFIRMED_DUPLICATE_SOURCE, CURRENT_DOMAIN_SOURCE, HARD_REVIEW_SOURCE,
    prepare_historical_metadata, build_historical_feature_dataset,
    split_historical_domain_aware, split_summary, historical_training_weights,
)
from src.evaluation.evaluation3_preview_active_learning import (
    attach_labels, copy_batch_previews, fit_models, select_resolution_batch,
    write_review_sheet, xframe,
)
from src.evaluation.evaluation3_active_learning import (
    DUPLICATE, NON_DUPLICATE, apply_triage, binary_metrics,
    choose_triage_thresholds,
)
from IPython.display import display
print('BRANCH:',BRANCH)


## 1. Load historical human labels + cached preview features

Cell này dùng lại cache đã tạo bởi NB11 historical; không scan toàn bộ EVALUATION3.


In [ ]:
DRIVE=Path('/content/drive/MyDrive')
ROOT_CANDIDATES=[DRIVE/'phash_ssim_threshold',DRIVE/'EVALUATION3'/'phash_ssim_threshold']
HIST_ROOT=next((p for p in ROOT_CANDIDATES if (p/'confirmed_duplicates_dedup.csv').is_file()),None)
if HIST_ROOT is None:
    raise FileNotFoundError('Không thấy phash_ssim_threshold trong MyDrive; hãy Add shortcut shared folder vào My Drive.')

HARD_REVIEW_SHEET_ID='1b9pj-E_C0BMfyN8s6upKxzcfntR_T77QAfLMwBDBu_g'
import google.auth, gspread
creds,_=google.auth.default(); gc=gspread.authorize(creds)
ws=gc.open_by_key(HARD_REVIEW_SHEET_ID).worksheet('hard_negative_review_BLIND')
hard_labels=pd.DataFrame(ws.get_all_records())
metadata=prepare_historical_metadata(historical_root=HIST_ROOT,hard_review_labels=hard_labels)

WORK_DIR=DRIVE/'evaluation3_active_learning_nb11_historical'
WORK_DIR.mkdir(parents=True,exist_ok=True)
FEATURE_CACHE=WORK_DIR/'historical_preview_features.csv'
features=build_historical_feature_dataset(metadata,historical_root=HIST_ROOT,cache_csv=FEATURE_CACHE,workers=4)
print('Rows:',len(features),'groups:',features.group_id.nunique())
display(pd.crosstab(features.source,features.human_label,margins=True))
print('Original hard labels:',features[features.source.eq(HARD_REVIEW_SOURCE)].human_label_original.value_counts().to_dict())
source_guess=features.source.eq(CONFIRMED_DUPLICATE_SOURCE).astype(int)
print('WARNING — source-only baseline accuracy:',float((source_guess.to_numpy()==features.target.to_numpy()).mean()))


## 2. Source-aware, group-disjoint split

Validation/test chỉ chứa hard-review rows. Nếu một E3 image nằm trong validation/test thì cả easy positive anchor cùng E3 image cũng bị loại khỏi train.


In [ ]:
RANDOM_STATE=42
train,val,test=split_historical_domain_aware(features,random_state=RANDOM_STATE)
display(split_summary(('train',train),('validation_hard_only',val),('test_hard_only',test)))
display(pd.concat([
    pd.crosstab(train.source,train.human_label).assign(split='train'),
    pd.crosstab(val.source,val.human_label).assign(split='validation'),
    pd.crosstab(test.source,test.human_label).assign(split='test'),
]).fillna(0))
assert val.source.eq(HARD_REVIEW_SOURCE).all() and test.source.eq(HARD_REVIEW_SOURCE).all()
assert set(train.group_id).isdisjoint(set(val.group_id))
assert set(train.group_id).isdisjoint(set(test.group_id))
assert set(val.group_id).isdisjoint(set(test.group_id))
print('Source-aware group leakage check: PASS')


## 3. Train weighted baselines + choose thresholds on hard validation

Easy confirmed positives có weight `0.25`; hard historical labels có weight `1.0`. Hai threshold bên dưới chỉ là **observed pilot thresholds**, không phải statistical guarantee vì positive hard examples còn ít.


In [ ]:
weights=historical_training_weights(train)
models,report=fit_models(train,val,RANDOM_STATE,sample_weight=weights)
display(report)
BEST_NAME=str(report.iloc[0].model); BEST_MODEL=models[BEST_NAME]
BEST_IDX=list(BEST_MODEL.classes_).index(1)
val_prob=BEST_MODEL.predict_proba(xframe(val))[:,BEST_IDX]
THRESHOLDS=choose_triage_thresholds(
    val.target.astype(int).to_numpy(),val_prob,
    target_auto_duplicate_precision=0.95,
    target_auto_non_npv=0.95,
    minimum_auto_examples=5,
)
print('BEST:',BEST_NAME)
print('TRIAGE:',THRESHOLDS)


## 4. Honest historical test (hard-review domain only)

Con số ở đây thay thế historical held-out 1.0 cũ. Chỉ đọc một lần cho pilot; đừng dùng test này để tiếp tục chỉnh model/threshold.


In [ ]:
from sklearn.metrics import confusion_matrix
test_prob=BEST_MODEL.predict_proba(xframe(test))[:,BEST_IDX]
print('rows / labels:',len(test),test.human_label.value_counts().to_dict())
print('binary metrics:',binary_metrics(test.target.astype(int).to_numpy(),test_prob))
print('confusion matrix [[TN,FP],[FN,TP]]:')
print(confusion_matrix(test.target.astype(int),test_prob>=0.5,labels=[0,1]))
test_triage=apply_triage(test_prob,THRESHOLDS)
print('triage:',pd.Series(test_triage).value_counts().to_dict())


## 5. Domain adaptation bằng các active-learning rounds hiện tại

`round_01_query` được chọn vì gần `p=0.5`, nên 30/30 MANUAL là điều được thiết kế sẵn—not a failed test. Cell này dùng các label đó để retrain, với current-domain weight `4.0`.


In [ ]:
FAST_DIR=DRIVE/'evaluation3_active_learning_nb11_fast'
fast_features=FAST_DIR/'preview_features_600.csv'
round_files=sorted(FAST_DIR.glob('round_*.xlsx')) if FAST_DIR.is_dir() else []
ADAPTED_READY=False
if not fast_features.is_file() or not round_files:
    print('SKIPPED: không thấy preview_features_600.csv hoặc round_*.xlsx trong',FAST_DIR)
else:
    current_pool=pd.read_csv(fast_features)
    current=attach_labels(current_pool,round_files)
    current=current[current.target.notna()].copy()
    print('Current active-learning labels:',len(current),current.human_label.value_counts().to_dict())
    if len(current)<10 or current.target.nunique()<2:
        print('CHƯA READY: cần ít nhất 10 current labels và có cả hai class.')
    else:
        preview_parent=Path(str(current_pool.iloc[0].preview_file)).parent.parent
        key_path=preview_parent/'evaluation3_manual_review_KEY.csv'
        if key_path.is_file():
            key=pd.read_csv(key_path)
            keep=[c for c in ['pair_id','e3_outfit_id','e3_slot','e3_image','polyvore_splits'] if c in key.columns]
            current_pool=current_pool.merge(key[keep].drop_duplicates('pair_id'),on='pair_id',how='left',validate='one_to_one')
            current=attach_labels(current_pool,round_files)
            current=current[current.target.notna()].copy()
            print('Recovered group metadata from:',key_path)
        else:
            print('WARNING: không thấy adjacent KEY; resolution queue sẽ fallback one-per-pair.')

        current['source']=CURRENT_DOMAIN_SOURCE
        current['group_id']=current['e3_outfit_id'].astype(str) if 'e3_outfit_id' in current.columns else current['pair_id'].astype(str)
        adapted_train=pd.concat([train,current],ignore_index=True,sort=False)
        adapted_weights=historical_training_weights(adapted_train)
        adapted_models,adapted_report=fit_models(adapted_train,val,RANDOM_STATE,sample_weight=adapted_weights)
        display(adapted_report)
        ADAPTED_NAME=str(adapted_report.iloc[0].model); ADAPTED_MODEL=adapted_models[ADAPTED_NAME]
        ADAPTED_IDX=list(ADAPTED_MODEL.classes_).index(1)
        adapted_val_prob=ADAPTED_MODEL.predict_proba(xframe(val))[:,ADAPTED_IDX]
        ADAPTED_THRESHOLDS=choose_triage_thresholds(
            val.target.astype(int).to_numpy(),adapted_val_prob,
            target_auto_duplicate_precision=0.95,target_auto_non_npv=0.95,
            minimum_auto_examples=5,
        )
        unlabeled=current_pool[~current_pool.pair_id.astype(str).isin(set(current.pair_id.astype(str)))].copy()
        if len(unlabeled):
            p=ADAPTED_MODEL.predict_proba(xframe(unlabeled))[:,ADAPTED_IDX]
            tri=apply_triage(p,ADAPTED_THRESHOLDS)
            print('Current unlabeled pool triage:',pd.Series(tri).value_counts().to_dict())
            print('Manual fraction:',float(np.mean(tri=='MANUAL_REVIEW')))
        ADAPTED_READY=True


## 6. Tạo resolution queue: one high-P(DUP) candidate per unresolved group

Queue này phục vụ mục tiêu hoàn tất contamination audit. Nó khác uncertainty query: ta muốn tìm **một duplicate càng sớm càng tốt** cho mỗi E3 group rồi không review các candidate dư thừa của group đó nữa.


In [ ]:
CREATE_RESOLUTION_BATCH=True
RESOLUTION_BATCH_SIZE=30
RESOLUTION_XLSX=FAST_DIR/'resolution_round_01.xlsx'
if not ADAPTED_READY or not CREATE_RESOLUTION_BATCH:
    print('SKIPPED')
elif RESOLUTION_XLSX.is_file():
    print('Đã có, không overwrite:',RESOLUTION_XLSX)
else:
    group_col='e3_outfit_id' if 'e3_outfit_id' in current_pool.columns else '__pair_fallback__'
    batch=select_resolution_batch(
        ADAPTED_MODEL,current_pool,batch_size=RESOLUTION_BATCH_SIZE,
        group_column=group_col,labeled_rows=current,
    )
    write_review_sheet(batch,RESOLUTION_XLSX)
    copy_batch_previews(batch,FAST_DIR/'resolution_round_01_previews')
    print('Created:',RESOLUTION_XLSX,'rows:',len(batch))
    display(batch[[c for c in ['pair_id','e3_outfit_id','model_probability_duplicate','rgb_ssim','edge_ssim','mean_lab_delta'] if c in batch.columns]])


## Cách đọc kết quả

- Historical score 1.0 cũ không còn được dùng làm bằng chứng vì source mixture quá dễ.
- 30/30 MANUAL ở `round_01` không phải lỗi: đó là uncertainty batch. Hãy nhìn **hard-only test**, full-pool triage và resolution yield.
- Nếu cùng sản phẩm nhưng khác ảnh/resize, ghi `SAME_PRODUCT_DIFFERENT_IMAGE`; notebook map về duplicate nhưng vẫn giữ subtype.
- Nếu same-product/different-color phải được xem là duplicate hay non-duplicate, team cần chốt policy riêng. Đừng buộc classifier học từ label không nhất quán.
- Nếu sau 2–3 current rounds hard-only metrics và resolution yield vẫn thấp, bước tiếp theo hợp lý là thêm DINO/FashionCLIP cosine như auxiliary features; không tiếp tục mò một ngưỡng SSIM mới.
